# RBI Payments — Phase 4: Forecasting

Pickup from the cleaned panel built in phase 3. Goal: 12-month forward forecast for UPI transaction value (Apr-2025 → Mar-2026), plus per-mode forecasts across all 6 payment modes.

Train/test split:
- **Train:** Nov-2019 → Dec-2022
- **Test:** Jan-2023 → Dec-2024 (24 months)
- **Forecast horizon:** Apr-2025 → Mar-2026

Three models compared against a naive seasonal baseline: SARIMA, Holt-Winters, Prophet. Primary model = best test MAPE that beats naive.

## Setup

In [ ]:
# run once if needed
# !pip install pmdarima prophet statsmodels scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats as stats

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima
from prophet import Prophet

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Data Loading

In [ ]:
# adjust path if running from a different working directory
RAW = pd.read_csv("../data/rbi_payments_clean.csv", parse_dates=["Date"])
RAW = RAW.sort_values(["Payment_Mode", "Date"]).reset_index(drop=True)

RAW.head()

In [ ]:
RAW.isnull().sum()

In [ ]:
RAW.describe()

In [ ]:
print(RAW["Payment_Mode"].unique())
print(f"Date range: {RAW['Date'].min().date()} to {RAW['Date'].max().date()}")
print(f"Total rows: {len(RAW)}")

In [ ]:
MODES      = ["UPI", "NEFT", "Credit", "Debit", "IMPS", "RTGS"]
FOCUS      = "UPI"
TRAIN_END  = "2022-12-31"
TEST_START = "2023-01-01"
TEST_END   = "2024-12-31"
HORIZON    = 12

# 15% uplift assumption for Credit-on-UPI scenario
# basis: CC CAGR FY19-25 ~18.5%; applying ~half the differential over UPI baseline
SCEN_B_UPLIFT = 0.15

# differencing orders from ADF+KPSS in EDA notebook
D_MAP = {"UPI": 2, "NEFT": 1, "Credit": 1, "Debit": 1, "IMPS": 1, "RTGS": 1}

# HW spec — UPI linear growth uses additive; card modes use multiplicative
HW_SPEC = {
    "UPI":    ("add", "add"),
    "NEFT":   ("add", "add"),
    "Credit": ("mul", "mul"),
    "Debit":  ("mul", "mul"),
    "IMPS":   ("add", "add"),
    "RTGS":   ("add", "add"),
}

_last  = RAW["Date"].max()
FC_IDX = pd.date_range(_last + pd.DateOffset(months=1), periods=HORIZON, freq="MS")
print(f"Forecast window: {FC_IDX[0].date()} to {FC_IDX[-1].date()}")

In [ ]:
def get_series(mode, col="Value"):
    return RAW[RAW["Payment_Mode"] == mode].set_index("Date")[col].sort_index()

def mape(a, p):
    a, p = np.asarray(a), np.asarray(p)
    return float(np.mean(np.abs((a - p) / np.where(a == 0, 1e-9, a))) * 100)

def rmse(a, p):
    return float(np.sqrt(mean_squared_error(np.asarray(a), np.asarray(p))))

def mae(a, p):
    return float(mean_absolute_error(np.asarray(a), np.asarray(p)))

def fmt_T(ax):
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x/1e12:.1f}T"))

## Feature Engineering

Most features built in phase 3. Confirming the  column is consistent before fitting SARIMA on log scale.

In [ ]:
# sanity check: log_value should equal log(Value)
upi = RAW[RAW["Payment_Mode"] == "UPI"].copy()
diff = (np.log(upi["Value"]) - upi["log_value"]).abs().max()
print(f"Max deviation: {diff:.6f}")  # should be ~0

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

upi_val = get_series(FOCUS, "Value")
axes[0].plot(upi_val.index, upi_val / 1e12)
axes[0].set_title("UPI monthly value (₹T)")
fmt_T(axes[0])

upi_log = get_series(FOCUS, "log_value")
axes[1].plot(upi_log.index, upi_log, color="steelblue")
axes[1].set_title("UPI log(Value) -- used for SARIMA")

plt.tight_layout()
plt.show()

### Stationarity

Full ADF+KPSS grid run in EDA notebook. UPI needs d=2 due to explosive growth. Confirming here.

In [ ]:
s = upi_log.dropna()

for d, label in [(0, "level"), (1, "1st diff"), (2, "2nd diff")]:
    if d == 0:
        s_d = s
    elif d == 1:
        s_d = s.diff().dropna()
    else:
        s_d = s.diff().diff().dropna()
    adf_p  = adfuller(s_d, autolag="AIC")[1]
    kpss_p = kpss(s_d, regression="c", nlags="auto")[1]
    ok = adf_p < 0.05 and kpss_p > 0.05
    print(f"  d={d} ({label}): ADF p={adf_p:.4f}  KPSS p={kpss_p:.4f}  {'-> stationary' if ok else '-> not stationary'}")

In [ ]:
# ACF/PACF -- level and first-differenced
NLAGS = 24
fig, axes = plt.subplots(2, 2, figsize=(13, 6))

for row, (label, s_plot) in enumerate([("Level", s), ("1st diff", s.diff().dropna())]):
    ci = 1.96 / np.sqrt(len(s_plot))
    for col, (vals, kind) in enumerate([(acf(s_plot, nlags=NLAGS, fft=True), "ACF"),
                                         (pacf(s_plot, nlags=NLAGS), "PACF")]):
        ax = axes[row][col]
        bar_c = ["steelblue" if abs(v) > ci else "lightgrey" for v in vals]
        ax.bar(range(len(vals)), vals, color=bar_c, width=0.4)
        ax.axhline(ci,  color="orange", ls="--", lw=1.2)
        ax.axhline(-ci, color="orange", ls="--", lw=1.2)
        ax.axhline(0, color="black", lw=0.7)
        ax.set_title(f"{kind} -- {label}")
        ax.set_xlabel("Lag (months)")

plt.tight_layout()
plt.savefig("acf_pacf_upi.png", dpi=120, bbox_inches="tight")
plt.show()

## Forecasting

### Naive seasonal baseline

Simplest possible forecast: replicate the same month from the prior year. All models must beat this.

In [ ]:
upi_train = upi_val[:TRAIN_END]
upi_test  = upi_val[TEST_START:TEST_END]

# y_hat(t) = y(t-12)
naive_pred = np.array([upi_train.iloc[-(12 - i % 12)] for i in range(len(upi_test))])

NAIVE_MAPE = mape(upi_test.values, naive_pred)
print(f"Naive MAPE: {NAIVE_MAPE:.2f}%")
print(f"RMSE: ₹{rmse(upi_test.values, naive_pred)/1e9:,.1f}Bn")

### Model 1: SARIMA

Fitting on log scale to stabilise variance.  with AIC selects order from SARIMA(0-2, d, 0-2)(0-1, 1, 0-1)[12]. Restricted to max_p/q=2 to avoid overfitting on ~65 obs. d=2 from stationarity tests.

Ljung-Box on residuals checks whether the model captured the serial structure or left autocorrelation behind.

In [ ]:
upi_log_train = get_series(FOCUS, "log_value")[:TRAIN_END]
upi_log_test  = get_series(FOCUS, "log_value")[TEST_START:TEST_END]

print("Searching SARIMA order space...")
sarima_cv = auto_arima(
    upi_log_train,
    d=D_MAP[FOCUS], D=1, m=12,
    start_p=0, max_p=2, start_q=0, max_q=2,
    start_P=0, max_P=1, start_Q=0, max_Q=1,
    information_criterion="aic",
    stepwise=True,
    suppress_warnings=True,
    error_action="ignore",
)
print(f"Selected: SARIMA{sarima_cv.order}x{sarima_cv.seasonal_order}")
print(f"AIC: {sarima_cv.aic():.2f}")

In [ ]:
resid = sarima_cv.resid()
lb = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
print(lb[["lb_stat", "lb_pvalue"]])
ljung_pass = all(lb["lb_pvalue"] > 0.05)
print(f"
White noise: {ljung_pass}")

In [ ]:
sarima_test_log = sarima_cv.predict(n_periods=len(upi_log_test))
sarima_test_val = np.exp(np.asarray(sarima_test_log))

SARIMA_MAPE = mape(upi_test.values, sarima_test_val)
print(f"MAPE:  {SARIMA_MAPE:.2f}%")
print(f"RMSE:  ₹{rmse(upi_test.values, sarima_test_val)/1e9:,.1f}Bn")
print(f"Beats naive: {SARIMA_MAPE < NAIVE_MAPE}")

In [ ]:
# residual diagnostics
resid_std = resid / resid.std()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(resid_std, lw=0.9, color="steelblue")
axes[0].axhline(0, color="black", lw=0.7)
axes[0].axhline(2,  color="orange", ls="--", lw=1)
axes[0].axhline(-2, color="orange", ls="--", lw=1)
axes[0].set_title("Standardised residuals")

x = np.linspace(-3.5, 3.5, 200)
axes[1].hist(resid_std, bins=15, color="steelblue", edgecolor="white", density=True)
axes[1].plot(x, stats.norm.pdf(x), color="orange", lw=1.5)
axes[1].set_title("Residual distribution")

acf_r = acf(resid, nlags=20, fft=True)
ci_r  = 1.96 / np.sqrt(len(resid))
axes[2].bar(range(len(acf_r)), acf_r,
            color=["steelblue" if abs(v) > ci_r else "lightgrey" for v in acf_r],
            width=0.5)
axes[2].axhline(ci_r,  color="orange", ls="--", lw=1)
axes[2].axhline(-ci_r, color="orange", ls="--", lw=1)
axes[2].set_title("ACF of residuals")

fig.suptitle(f"SARIMA{sarima_cv.order}x{sarima_cv.seasonal_order} -- residual diagnostics")
plt.tight_layout()
plt.savefig("sarima_residuals.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# production refit on full data
upi_log_full = get_series(FOCUS, "log_value")

sarima_prod = SARIMAX(
    upi_log_full,
    order=sarima_cv.order,
    seasonal_order=sarima_cv.seasonal_order,
    trend="c",
).fit(disp=False)

fc_obj = sarima_prod.get_forecast(steps=HORIZON)
fc_df  = fc_obj.summary_frame(alpha=0.05)

SARIMA_FC   = np.exp(fc_df["mean"].values)
SARIMA_LO95 = np.exp(fc_df["mean_ci_lower"].values)
SARIMA_HI95 = np.exp(fc_df["mean_ci_upper"].values)
_shrink = 1.282 / 1.96
SARIMA_LO80 = SARIMA_FC - _shrink * (SARIMA_FC - SARIMA_LO95)
SARIMA_HI80 = SARIMA_FC + _shrink * (SARIMA_HI95 - SARIMA_FC)

print(f"Forecast range: ₹{SARIMA_FC.min()/1e12:.2f}T -- ₹{SARIMA_FC.max()/1e12:.2f}T")

### Model 2: Holt-Winters

UPI shows near-linear growth, so additive trend + additive seasonal (ETS A,A,A) is the right spec. Tried multiplicative first -- it collapsed with alpha→1.0 and failed to converge on the short training window.

For Credit and Debit cards the multiplicative spec is appropriate since growth is percentage-driven.

In [ ]:
hw_cv = ExponentialSmoothing(
    upi_train,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
).fit(optimized=True, use_brute=True)

print(f"alpha (level):    {hw_cv.params['smoothing_level']:.4f}")
print(f"beta  (trend):    {hw_cv.params['smoothing_trend']:.4f}")
print(f"gamma (seasonal): {hw_cv.params['smoothing_seasonal']:.4f}")

In [ ]:
hw_test_pred = hw_cv.forecast(len(upi_test)).values

HW_MAPE = mape(upi_test.values, hw_test_pred)
print(f"MAPE: {HW_MAPE:.2f}%")
print(f"RMSE: ₹{rmse(upi_test.values, hw_test_pred)/1e9:,.1f}Bn")
print(f"Beats naive: {HW_MAPE < NAIVE_MAPE}")

In [ ]:
hw_prod = ExponentialSmoothing(
    upi_val,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
).fit(optimized=True, use_brute=True)

HW_FC = hw_prod.forecast(HORIZON).values

# approximate CIs from residual std (HW has no analytic intervals)
_hw_std = np.std(upi_val.values - hw_prod.fittedvalues.values)
_h = np.arange(1, HORIZON + 1)
HW_LO95 = HW_FC - 1.96  * _hw_std * np.sqrt(_h)
HW_HI95 = HW_FC + 1.96  * _hw_std * np.sqrt(_h)
HW_LO80 = HW_FC - 1.282 * _hw_std * np.sqrt(_h)
HW_HI80 = HW_FC + 1.282 * _hw_std * np.sqrt(_h)

print(f"Forecast range: ₹{HW_FC.min()/1e12:.2f}T -- ₹{HW_FC.max()/1e12:.2f}T")

### Model 3: Prophet

Adding two binary regressors:  and . These enter as additional linear components on top of trend+seasonality.

Multiplicative seasonality because UPI seasonal swings are growing proportionally with the trend.  is higher than default (0.05) to let the model track UPI's post-2022 acceleration. One concern: aggressive changepoint placement near the end of training may cause the trend to be overfit to a short recent window, which would inflate test-window errors at H+18 to H+24.

In [ ]:
prop_df = (
    RAW[RAW["Payment_Mode"] == FOCUS]
    [["Date", "Value", "is_covid_shock", "is_policy_change"]]
    .rename(columns={"Date": "ds", "Value": "y"})
    .sort_values("ds")
)

prop_train = prop_df[prop_df["ds"] <= TRAIN_END]
prop_test  = prop_df[(prop_df["ds"] >= TEST_START) & (prop_df["ds"] <= TEST_END)]

m_cv = Prophet(
    changepoint_prior_scale=0.3,
    seasonality_mode="multiplicative",
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.95,
)
m_cv.add_regressor("is_covid_shock")
m_cv.add_regressor("is_policy_change")
m_cv.fit(prop_train)

In [ ]:
fc_test_prop = m_cv.predict(prop_test[["ds", "is_covid_shock", "is_policy_change"]])

PROPHET_MAPE = mape(prop_test["y"].values, fc_test_prop["yhat"].values)
print(f"MAPE: {PROPHET_MAPE:.2f}%")
print(f"Beats naive: {PROPHET_MAPE < NAIVE_MAPE}")

In [ ]:
# visual sanity check on test window
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(upi_val.index, upi_val / 1e12, label="actual", lw=1.5)
ax.plot(pd.to_datetime(fc_test_prop["ds"]), fc_test_prop["yhat"] / 1e12,
        ls="--", color="orange", label="prophet test")
fmt_T(ax)
ax.legend()
ax.set_title("Prophet test-window fit")
plt.tight_layout()
plt.show()

In [ ]:
# production refit
m_prod = Prophet(
    changepoint_prior_scale=0.3,
    seasonality_mode="multiplicative",
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.95,
)
m_prod.add_regressor("is_covid_shock")
m_prod.add_regressor("is_policy_change")
m_prod.fit(prop_df)

fut = pd.DataFrame({
    "ds": FC_IDX,
    "is_covid_shock": [0] * HORIZON,
    "is_policy_change": [1] * HORIZON,  # credit-on-UPI active
})
fc_prod_prop = m_prod.predict(fut)

PROPHET_FC   = fc_prod_prop["yhat"].values
PROPHET_LO95 = fc_prod_prop["yhat_lower"].values
PROPHET_HI95 = fc_prod_prop["yhat_upper"].values
PROPHET_LO80 = PROPHET_FC - (1.282/1.96) * (PROPHET_FC - PROPHET_LO95)
PROPHET_HI80 = PROPHET_FC + (1.282/1.96) * (PROPHET_HI95 - PROPHET_FC)

print(f"Forecast range: ₹{PROPHET_FC.min()/1e12:.2f}T -- ₹{PROPHET_FC.max()/1e12:.2f}T")

## Evaluation

In [ ]:
eval_rows = [
    ("Naive seasonal",          NAIVE_MAPE,   rmse(upi_test.values, naive_pred),         mae(upi_test.values, naive_pred)),
    (f"SARIMA{sarima_cv.order}", SARIMA_MAPE,  rmse(upi_test.values, sarima_test_val),    mae(upi_test.values, sarima_test_val)),
    ("Holt-Winters ETS(A,A,A)", HW_MAPE,      rmse(upi_test.values, hw_test_pred),       mae(upi_test.values, hw_test_pred)),
    ("Prophet",                  PROPHET_MAPE, rmse(prop_test["y"].values, fc_test_prop["yhat"].values),
                                               mae(prop_test["y"].values, fc_test_prop["yhat"].values)),
]

eval_df = pd.DataFrame(eval_rows, columns=["Model", "MAPE (%)", "RMSE (₹)", "MAE (₹)"])
eval_df["MAPE (%)"] = eval_df["MAPE (%)"].round(2)
eval_df["RMSE (₹ Cr)"] = (eval_df["RMSE (₹)"] / 1e7).round(0).astype(int)
eval_df["MAE (₹ Cr)"]  = (eval_df["MAE (₹)"]  / 1e7).round(0).astype(int)
eval_df["beats_naive"]     = eval_df["MAPE (%)"] < NAIVE_MAPE
eval_df = eval_df.drop(columns=["RMSE (₹)", "MAE (₹)"]).sort_values("MAPE (%)")

print(eval_df.to_string(index=False))

In [ ]:
best_row   = eval_df[eval_df["beats_naive"]].iloc[0]
best_model = best_row["Model"]
print(f"Primary model: {best_model}  (MAPE={best_row['MAPE (%)']}%)")

if "SARIMA" in best_model:
    PRIMARY_FC, PRIMARY_LO95, PRIMARY_HI95 = SARIMA_FC, SARIMA_LO95, SARIMA_HI95
    PRIMARY_LO80, PRIMARY_HI80 = SARIMA_LO80, SARIMA_HI80
    PRIMARY_LABEL = best_model
elif "Holt" in best_model:
    PRIMARY_FC, PRIMARY_LO95, PRIMARY_HI95 = HW_FC, HW_LO95, HW_HI95
    PRIMARY_LO80, PRIMARY_HI80 = HW_LO80, HW_HI80
    PRIMARY_LABEL = "Holt-Winters ETS(A,A,A)"
else:
    PRIMARY_FC, PRIMARY_LO95, PRIMARY_HI95 = PROPHET_FC, PROPHET_LO95, PROPHET_HI95
    PRIMARY_LO80, PRIMARY_HI80 = PROPHET_LO80, PROPHET_HI80
    PRIMARY_LABEL = "Prophet"

**Notes:**

SARIMA and Holt-Winters both consistently outperform naive. Prophet is competitive in-sample but often underperforms at the tail of the test window. The changepoint detection places too many changepoints in late 2022 when UPI growth was accelerating, which causes an overoptimistic trend that decays incorrectly around H+18 to H+24.

Holt-Winters is more interpretable for a linear growth series. SARIMA's advantage is in capturing seasonal AR/MA structure, at the cost of wider CIs from log back-transform.

Neither SARIMA nor HW includes the Credit-on-UPI policy as an explicit regressor -- that is handled as Scenario B.

### Scenarios

Scenario A: primary model output, no extra uplift.
Scenario B: +15% over Scenario A, proxying value uplift from Credit-on-UPI rollout.

The 15% is a working assumption based on CC CAGR differential, not an endogenous model. Treat it as a sensitivity.

In [ ]:
SCEN_A_FC   = PRIMARY_FC.copy()
SCEN_A_LO95 = PRIMARY_LO95.copy()
SCEN_A_HI95 = PRIMARY_HI95.copy()
SCEN_A_LO80 = PRIMARY_LO80.copy()
SCEN_A_HI80 = PRIMARY_HI80.copy()

SCEN_B_FC   = SCEN_A_FC   * (1 + SCEN_B_UPLIFT)
SCEN_B_LO95 = SCEN_A_LO95 * (1 + SCEN_B_UPLIFT * 0.55)
SCEN_B_HI95 = SCEN_A_HI95 * (1 + SCEN_B_UPLIFT * 1.45)

for dt, a, b in zip(FC_IDX, SCEN_A_FC, SCEN_B_FC):
    print(f"{dt.strftime('%b %Y')}: Scen A = ₹{a/1e12:.2f}T | Scen B = ₹{b/1e12:.2f}T")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(upi_val.index, upi_val / 1e12, color="steelblue", lw=2, label="UPI actual")
ax.axvspan(pd.Timestamp(TEST_START), pd.Timestamp(TEST_END), alpha=0.06, color="salmon", label="Test window")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-10-01"), alpha=0.08, color="grey")
ax.text(pd.Timestamp("2020-05-01"), upi_val.max() * 0.35 / 1e12, "COVID", fontsize=8, color="grey")

ax.fill_between(FC_IDX, SCEN_A_LO95/1e12, SCEN_A_HI95/1e12, alpha=0.12, color="darkorange")
ax.fill_between(FC_IDX, SCEN_A_LO80/1e12, SCEN_A_HI80/1e12, alpha=0.22, color="darkorange")
ax.plot(FC_IDX, SCEN_A_FC/1e12,  color="darkorange", lw=2, label=f"Scen A -- {PRIMARY_LABEL}")
ax.plot(FC_IDX, SCEN_B_FC/1e12,  color="purple",     lw=2, ls="--",
        label=f"Scen B -- +{SCEN_B_UPLIFT*100:.0f}% (Credit-on-UPI)")
ax.axvline(_last, color="navy", ls=":", lw=1.2)

fmt_T(ax)
ax.set_ylabel("₹ Trillion / month")
ax.set_title(f"UPI Value Forecast Apr 2025 -- Mar 2026 | {PRIMARY_LABEL}")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("upi_forecast.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(upi_val.index, upi_val/1e12, color="steelblue", lw=2, label="actual")
ax.plot(upi_test.index, sarima_test_val/1e12, ls="--", lw=1.6,
        label=f"SARIMA (MAPE={SARIMA_MAPE:.1f}%)")
ax.plot(upi_test.index, hw_test_pred/1e12, ls="-.", lw=1.6,
        label=f"Holt-Winters (MAPE={HW_MAPE:.1f}%)")
ax.plot(pd.to_datetime(fc_test_prop["ds"]), fc_test_prop["yhat"]/1e12, ls=":", lw=1.6,
        label=f"Prophet (MAPE={PROPHET_MAPE:.1f}%)")
ax.plot(upi_test.index, naive_pred/1e12, color="red", lw=1.2, alpha=0.6,
        label=f"Naive (MAPE={NAIVE_MAPE:.1f}%)")

ax.axvspan(pd.Timestamp(TEST_START), pd.Timestamp(TEST_END), alpha=0.05, color="salmon")
fmt_T(ax)
ax.set_title("Model comparison -- 24-month test window (Jan 2023 -- Dec 2024)")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

### All-modes forecast

In [ ]:
_mode_uplift = {"UPI": 0.15, "NEFT": 0.03, "Credit": 0.12, "Debit": 0.02, "IMPS": 0.05, "RTGS": 0.02}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("12-month value forecast by payment mode (Apr 2025 -- Mar 2026)", fontsize=12)

ALL_MODES_FC = {}

for ax, mode in zip(axes.flatten(), MODES):
    s_log  = get_series(mode, "log_value")
    s_val  = get_series(mode, "Value")
    uplift = _mode_uplift[mode]

    try:
        if mode == FOCUS:
            fc_val, fc_lo95, fc_hi95 = SARIMA_FC, SARIMA_LO95, SARIMA_HI95
            order_label = f"SARIMA{sarima_cv.order}"
        else:
            aa = auto_arima(
                s_log, d=D_MAP[mode], D=1, m=12,
                start_p=0, max_p=2, start_q=0, max_q=2,
                start_P=0, max_P=1, start_Q=0, max_Q=1,
                stepwise=True, suppress_warnings=True, error_action="ignore",
            )
            sm = SARIMAX(s_log, order=aa.order, seasonal_order=aa.seasonal_order, trend="c").fit(disp=False)
            fc_df_m = sm.get_forecast(steps=HORIZON).summary_frame(alpha=0.05)
            fc_val  = np.exp(fc_df_m["mean"].values)
            fc_lo95 = np.exp(fc_df_m["mean_ci_lower"].values)
            fc_hi95 = np.exp(fc_df_m["mean_ci_upper"].values)
            order_label = f"SARIMA{aa.order}"

    except Exception as e:
        print(f"{mode}: SARIMA failed ({e}), using HW fallback")
        t_spec, s_spec = HW_SPEC[mode]
        try:
            hw_fb  = ExponentialSmoothing(s_val, trend=t_spec, seasonal=s_spec,
                                           seasonal_periods=12).fit(optimized=True, use_brute=True)
            fc_val = hw_fb.forecast(HORIZON).values
        except Exception:
            fc_val = np.full(HORIZON, s_val.iloc[-1])
        fc_lo95 = fc_val * 0.88
        fc_hi95 = fc_val * 1.12
        order_label = "HW fallback"

    fc_val_B = fc_val * (1 + uplift)
    ALL_MODES_FC[mode] = {"A": fc_val, "B": fc_val_B, "lo95": fc_lo95, "hi95": fc_hi95}

    ax.plot(s_val.index, s_val/1e12, lw=1.5)
    ax.fill_between(FC_IDX, fc_lo95/1e12, fc_hi95/1e12, alpha=0.15)
    ax.plot(FC_IDX, fc_val/1e12,   lw=2,   label="Scen A")
    ax.plot(FC_IDX, fc_val_B/1e12, lw=1.5, ls="--", label=f"Scen B (+{uplift*100:.0f}%)")
    ax.axvline(_last, color="grey", ls=":", lw=0.8)
    ax.set_title(f"{mode}  [{order_label}]")
    fmt_T(ax)
    ax.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig("all_modes_forecast.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for mode in MODES:
    fc = ALL_MODES_FC[mode]
    rows.append({
        "Mode":           mode,
        "Scen A min (₹T)": round(fc["A"].min() / 1e12, 2),
        "Scen A max (₹T)": round(fc["A"].max() / 1e12, 2),
        "Scen B max (₹T)": round(fc["B"].max() / 1e12, 2),
    })
pd.DataFrame(rows)

### Potential issues

- **Leakage risk:**  in Prophet forecast assumes the policy effect persists into FY26. Worth stress-testing with  to see how sensitive Scenario A is.
- **CI width:** SARIMA CIs on log scale back-transform multiplicatively, so they expand faster than expected for a near-linear series. HW CIs are approximate (residual std) and likely too narrow at H>6.
- **RTGS/NEFT:** both show structural plateau or decline. A linear trend component may not be appropriate here -- segmented trend or structural break model might be worth exploring.
- **No cross-mode dynamics:** UPI--NEFT substitution is visible in the data but not modelled. A VAR could improve NEFT/IMPS forecasts but probably overkill for this phase.

Phase 5 covers Power BI visualisation of these outputs.